# GIS 01 -- Spatial Fundamentals: Coordinates, Projections & Spatial Indexing

> **Geo-Instructor . GIS/Spatial Career Track . Notebook 1 of 3**

The Earth is a sphere. Screens are flat. Every map involves a tradeoff.
This notebook covers the math from WGS84 to Mercator to tile numbers to quadtrees.

```
Runtime: ~4 min  |  No GPU  |  numpy + matplotlib only
```

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
from mpl_toolkits.mplot3d import Axes3D
np.random.seed(0)
plt.rcParams.update({'figure.facecolor':'#F4F2F0','axes.facecolor':'#F4F2F0','font.family':'monospace','axes.grid':True,'grid.alpha':0.3})
print('Ready.')

---
## Part 1 -- The Earth as an Ellipsoid (WGS84)

GPS coordinates use WGS84: the Earth modeled as an oblate ellipsoid.
```
  Semi-major axis a = 6,378,137 m    (equatorial radius)
  Flattening     f = 1/298.257223563
  Semi-minor axis b = a*(1-f)        (polar radius)

Geodetic -> ECEF (Earth-Centered, Earth-Fixed):
  N = a / sqrt(1 - e^2 * sin^2(lat))     (prime vertical radius)
  X = (N + alt) * cos(lat) * cos(lon)
  Y = (N + alt) * cos(lat) * sin(lon)
  Z = (N*(1-e^2) + alt) * sin(lat)
```

In [ ]:
a = 6_378_137.0      # semi-major axis (m)
f = 1/298.257223563  # flattening
b = a * (1 - f)      # semi-minor axis
e2 = 2*f - f**2      # eccentricity squared

def geodetic_to_ecef(lat_deg, lon_deg, alt=0):
    lat = np.radians(lat_deg); lon = np.radians(lon_deg)
    N = a / np.sqrt(1 - e2 * np.sin(lat)**2)
    X = (N + alt) * np.cos(lat) * np.cos(lon)
    Y = (N + alt) * np.cos(lat) * np.sin(lon)
    Z = (N*(1-e2) + alt) * np.sin(lat)
    return np.array([X, Y, Z])

# World cities
cities = {
    'Tokyo':     (35.6762,  139.6503),
    'New York':  (40.7128,  -74.0060),
    'London':    (51.5074,   -0.1278),
    'Sydney':    (-33.8688, 151.2093),
    'Sao Paulo': (-23.5505, -46.6333),
    'Dubai':     (25.2048,   55.2708),
}
print('City                   | ECEF X(km)  | ECEF Y(km) | ECEF Z(km)')
for city, (lat,lon) in cities.items():
    X,Y,Z = geodetic_to_ecef(lat,lon)
    print(f'{city:22s} | {X/1000:10.0f}  | {Y/1000:9.0f} | {Z/1000:9.0f}')

# Plot on 3D sphere
fig = plt.figure(figsize=(9,7))
ax = fig.add_subplot(111, projection='3d')
u = np.linspace(0, 2*np.pi, 60); v = np.linspace(0, np.pi, 40)
xs = a/1e6*np.outer(np.cos(u), np.sin(v)); ys = a/1e6*np.outer(np.sin(u), np.sin(v)); zs = b/1e6*np.outer(np.ones_like(u), np.cos(v))
ax.plot_surface(xs, ys, zs, alpha=0.08, color='steelblue')
for city, (lat,lon) in cities.items():
    X,Y,Z = geodetic_to_ecef(lat,lon)/1e6
    ax.scatter(X,Y,Z,s=60,zorder=5)
    ax.text(X*1.05,Y*1.05,Z*1.05,city,fontsize=7)
ax.set_title('World cities on WGS84 ellipsoid')
ax.set_xlabel('X (Mm)'); ax.set_ylabel('Y (Mm)'); ax.set_zlabel('Z (Mm)')
plt.tight_layout(); plt.show()
print(f'WGS84: a={a:.0f}m, b={b:.0f}m, flattening={f:.8f}')

---
## Part 2 -- Mercator Projection

The Mercator projection maps a sphere to a flat rectangle:
```
  x = R * lon                       (linear)
  y = R * ln(tan(pi/4 + lat/2))     (nonlinear -- stretches toward poles)
```
Conformal (preserves angles/shapes) but NOT equal-area (distorts size).

In [ ]:
R = 6_371_000.0  # Earth mean radius

def mercator(lat_deg, lon_deg):
    lat = np.radians(np.clip(lat_deg, -85, 85))
    lon = np.radians(lon_deg)
    x = R * lon
    y = R * np.log(np.tan(np.pi/4 + lat/2))
    return x, y

# Plot world outline (approximated as lat/lon grid)
fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(14, 5))

# Tissot's indicatrix: circles at uniform lat/lon spacing
for lat in range(-60, 90, 30):
    for lon in range(-150, 180, 60):
        theta = np.linspace(0, 2*np.pi, 40)
        r = 8.0  # degrees radius
        lats = lat + r*np.sin(theta); lons = lon + r*np.cos(theta)
        mx, my = mercator(lats, lons)
        ax1.plot(mx/1e6, my/1e6, 'steelblue', lw=1, alpha=0.7)

# Graticule
for lat in range(-80, 90, 20):
    lons = np.linspace(-180, 180, 200)
    mx, my = mercator(lat, lons)
    ax1.plot(mx/1e6, my/1e6, color='gray', lw=0.4, alpha=0.5)
for lon in range(-180, 181, 20):
    lats = np.linspace(-85, 85, 100)
    mx, my = mercator(lats, lon)
    ax1.plot(mx/1e6, my/1e6, color='gray', lw=0.4, alpha=0.5)

ax1.set_aspect('equal'); ax1.set_title('Mercator with Tissot indicatrix\n(circles appear larger at high latitudes)')
ax1.set_xlabel('X (Mm)'); ax1.set_ylabel('Y (Mm)')

# Area distortion: Greenland vs Africa
def approx_area_deg2(lat1,lat2,lon1,lon2):
    lat_mid = np.radians((lat1+lat2)/2)
    dlat = np.radians(abs(lat2-lat1)); dlon = np.radians(abs(lon2-lon1))
    return (R*dlat) * (R*np.cos(lat_mid)*dlon) / 1e12  # million km^2

greenland_true = approx_area_deg2(60, 84, -58, -12)
africa_true    = approx_area_deg2(-35, 37, -20, 52)

# Mercator pixel area
def mercator_area(lat1,lat2,lon1,lon2):
    x1,y1=mercator(lat1,lon1); x2,y2=mercator(lat2,lon2)
    return abs((x2-x1)*(y2-y1)) / 1e12

greenland_merc = mercator_area(60,84,-58,-12)
africa_merc    = mercator_area(-35,37,-20,52)

ax2.bar(['Greenland\n(true)','Africa\n(true)','Greenland\n(Mercator)','Africa\n(Mercator)'],
        [greenland_true, africa_true, greenland_merc, africa_merc],
        color=['steelblue','steelblue','tomato','tomato'])
ax2.set_title('Area distortion: Mercator exaggerates Greenland\nvs Africa'); ax2.set_ylabel('Relative area (Mkm^2)')
for i,(v) in enumerate([greenland_true,africa_true,greenland_merc,africa_merc]):
    ax2.text(i, v+0.1, f'{v:.1f}', ha='center', fontsize=9)
plt.tight_layout(); plt.show()
print(f'True ratio Africa/Greenland: {africa_true/greenland_true:.1f}x')

In [ ]:
def haversine(lat1,lon1,lat2,lon2,R=6371e3):
    """Great-circle distance in meters."""
    lat1,lon1,lat2,lon2 = map(np.radians,[lat1,lon1,lat2,lon2])
    dlat=(lat2-lat1)/2; dlon=(lon2-lon1)/2
    a = np.sin(dlat)**2 + np.cos(lat1)*np.cos(lat2)*np.sin(dlon)**2
    return 2*R*np.arcsin(np.sqrt(a))

city_list = list(cities.items())
print('Great-circle distances (km):')
print(f'  Tokyo  -> New York: {haversine(35.68,139.65,40.71,-74.01)/1000:.0f} km')
print(f'  London -> Sydney:   {haversine(51.51,-0.13,-33.87,151.21)/1000:.0f} km')
print(f'  Dubai  -> SaoPaulo: {haversine(25.20,55.27,-23.55,-46.63)/1000:.0f} km')

# Show great circle path on Mercator (it curves!)
fig, ax = plt.subplots(figsize=(13, 5))
for lat in range(-80, 90, 20):
    lons=np.linspace(-180,180,200); mx,my=mercator(lat,lons)
    ax.plot(mx/1e6,my/1e6,color='gray',lw=0.3,alpha=0.4)
for lon in range(-180,181,20):
    lats=np.linspace(-85,85,100); mx,my=mercator(lats,lon)
    ax.plot(mx/1e6,my/1e6,color='gray',lw=0.3,alpha=0.4)

# Great circle Tokyo -> New York (interpolate along great circle)
for (n1,c1),(n2,c2) in [( ('Tokyo',(35.68,139.65)), ('New York',(40.71,-74.01)) ),
                         ( ('London',(51.51,-0.13)), ('Sydney',(-33.87,151.21)) )]:
    ns=100; lats=np.zeros(ns); lons=np.zeros(ns)
    p1=np.radians(c1); p2=np.radians(c2)
    v1=np.array([np.cos(p1[0])*np.cos(p1[1]),np.cos(p1[0])*np.sin(p1[1]),np.sin(p1[0])])
    v2=np.array([np.cos(p2[0])*np.cos(p2[1]),np.cos(p2[0])*np.sin(p2[1]),np.sin(p2[0])])
    for i,t in enumerate(np.linspace(0,1,ns)):
        v=v1*(1-t)+v2*t; v/=np.linalg.norm(v)
        lats[i]=np.degrees(np.arcsin(v[2]))
        lons[i]=np.degrees(np.arctan2(v[1],v[0]))
    mx,my=mercator(lats,lons)
    ax.plot(mx/1e6,my/1e6,'tomato',lw=2,label=f'{n1}->{n2}')
    mx0,my0=mercator(c1[0],c1[1]); mx1,my1=mercator(c2[0],c2[1])
    ax.plot([mx0/1e6,mx1/1e6],[my0/1e6,my1/1e6],'steelblue',lw=1,ls='--',alpha=0.6)
ax.legend(fontsize=8,title='red=great circle, blue=straight line')
ax.set_aspect('equal'); ax.set_title('Great circle paths look curved on Mercator\n(they are the shortest path on the sphere)')
plt.tight_layout(); plt.show()

---
## Part 3 -- Quadtree Spatial Indexing

A linear scan over 100M features takes seconds. Spatial indexes are essential.
The **quadtree** recursively subdivides 2D space into 4 quadrants,
letting you find all points in a bounding box in O(log N + k) instead of O(N).

In [ ]:
class Quadtree:
    MAX_POINTS = 8
    def __init__(self, bounds, depth=0, max_depth=8):
        self.bounds = bounds  # (minx,miny,maxx,maxy)
        self.points = []
        self.children = None
        self.depth = depth
        self.max_depth = max_depth

    def insert(self, pt):
        x,y = pt
        mx,my,Mx,My = self.bounds
        if not (mx<=x<=Mx and my<=y<=My): return False
        if self.children is None:
            self.points.append(pt)
            if len(self.points) > self.MAX_POINTS and self.depth < self.max_depth:
                self._split()
        else:
            for c in self.children: c.insert(pt)
        return True

    def _split(self):
        mx,my,Mx,My = self.bounds
        cx,cy = (mx+Mx)/2, (my+My)/2
        d = self.depth+1
        self.children = [
            Quadtree((mx,my,cx,cy),d,self.max_depth),
            Quadtree((cx,my,Mx,cy),d,self.max_depth),
            Quadtree((mx,cy,cx,My),d,self.max_depth),
            Quadtree((cx,cy,Mx,My),d,self.max_depth),
        ]
        for p in self.points:
            for c in self.children: c.insert(p)
        self.points = []

    def query_bbox(self, qbounds):
        mx,my,Mx,My = self.bounds
        qx,qy,qX,qY = qbounds
        if qX<mx or qx>Mx or qY<my or qy>My: return []
        result = [p for p in self.points if qx<=p[0]<=qX and qy<=p[1]<=qY]
        if self.children:
            for c in self.children: result.extend(c.query_bbox(qbounds))
        return result

    def all_bounds(self, acc=None):
        if acc is None: acc=[]
        if self.children:
            acc.append(self.bounds)
            for c in self.children: c.all_bounds(acc)
        return acc

import time
np.random.seed(1)
pts = np.random.uniform(0, 100, (10000, 2))
qt = Quadtree((0,0,100,100))
for p in pts: qt.insert(tuple(p))

query = (30, 30, 60, 60)
t0=time.time(); results_qt=qt.query_bbox(query); qt_time=time.time()-t0
t0=time.time(); results_bf=[p for p in pts if query[0]<=p[0]<=query[2] and query[1]<=p[1]<=query[3]]; bf_time=time.time()-t0
print(f'Quadtree: {len(results_qt)} pts found in {qt_time*1000:.2f} ms')
print(f'Brute force: {len(results_bf)} pts found in {bf_time*1000:.2f} ms')

fig, (ax1,ax2) = plt.subplots(1,2,figsize=(13,6))
for b in qt.all_bounds():
    mx,my,Mx,My=b; rect=plt.Rectangle((mx,my),Mx-mx,My-my,fill=False,edgecolor='steelblue',lw=0.4,alpha=0.5)
    ax1.add_patch(rect)
ax1.scatter(pts[:,0],pts[:,1],s=1,c='gray',alpha=0.3)
r=np.array(results_qt) if results_qt else np.zeros((0,2))
if len(r): ax1.scatter(r[:,0],r[:,1],s=8,c='tomato',zorder=4,label=f'Query hits ({len(r)})')
qr=plt.Rectangle((query[0],query[1]),query[2]-query[0],query[3]-query[1],fill=False,edgecolor='tomato',lw=2)
ax1.add_patch(qr); ax1.set_title('Quadtree decomposition + bbox query'); ax1.legend(fontsize=8)
ax2.bar(['Quadtree','Brute force'],[qt_time*1000,bf_time*1000],color=['steelblue','tomato'])
ax2.set_ylabel('Time (ms)'); ax2.set_title(f'Query time: {bf_time/max(qt_time,1e-9):.1f}x speedup')
plt.tight_layout(); plt.show()

---
## Part 4 -- Slippy Map Tile System

Google Maps, OpenStreetMap, Mapbox all use the XYZ tile system:
- At zoom 0: 1 tile covers the whole world
- At zoom N: 2^N x 2^N tiles, each 256x256 pixels
- Each tile addressed by (z, x, y) where y=0 is top (north)

```
  tile_x = floor((lon + 180) / 360 * 2^z)
  tile_y = floor((1 - ln(tan(lat_rad) + sec(lat_rad)) / pi) / 2 * 2^z)
```

In [ ]:
def lat_lon_to_tile(lat_deg, lon_deg, zoom):
    n = 2**zoom
    x = int((lon_deg + 180) / 360 * n)
    lat_r = np.radians(lat_deg)
    y = int((1 - np.log(np.tan(lat_r) + 1/np.cos(lat_r)) / np.pi) / 2 * n)
    return x, y

def tile_to_bbox(tx, ty, zoom):
    """Return (west, south, east, north) lat/lon bounds of a tile."""
    n = 2**zoom
    def tiley_to_lat(ty):
        return np.degrees(np.arctan(np.sinh(np.pi * (1 - 2*ty/n))))
    west  = tx/n * 360 - 180
    east  = (tx+1)/n * 360 - 180
    north = tiley_to_lat(ty)
    south = tiley_to_lat(ty+1)
    return west, south, east, north

# Tokyo
lat, lon = 35.6762, 139.6503
print(f'Tokyo ({lat}, {lon})')
print(f'Zoom | tile (x,y)          | bbox (approx km wide)')
for z in [0, 2, 5, 8, 10, 12, 15]:
    tx, ty = lat_lon_to_tile(lat, lon, z)
    w,s,e,n = tile_to_bbox(tx, ty, z)
    width_km = haversine(lat, w, lat, e) / 1000
    print(f'  {z:2d} | ({tx:6d}, {ty:6d}) | ~{width_km:.1f} km wide')

# Visualize tile grid at zoom 3
fig, ax = plt.subplots(figsize=(10, 6))
for z_show in [3]:
    n = 2**z_show
    for tx in range(n):
        for ty in range(n):
            w,s,e,no = tile_to_bbox(tx,ty,z_show)
            rect=plt.Rectangle((w,s),e-w,no-s,fill=False,edgecolor='steelblue',lw=0.4,alpha=0.6)
            ax.add_patch(rect)
for city,(clat,clon) in cities.items():
    ax.plot(clon,clat,'o',ms=5,color='tomato')
    ax.text(clon+1,clat,city,fontsize=7)
tx0,ty0 = lat_lon_to_tile(lat,lon,3); w,s,e,no=tile_to_bbox(tx0,ty0,3)
rect2=plt.Rectangle((w,s),e-w,no-s,facecolor='tomato',alpha=0.2,edgecolor='tomato',lw=2)
ax.add_patch(rect2)
ax.set_xlim(-180,180); ax.set_ylim(-90,90); ax.set_aspect('equal')
ax.set_title('Slippy map tiles at zoom 3 -- Tokyo tile highlighted'); ax.set_xlabel('Longitude'); ax.set_ylabel('Latitude')
plt.tight_layout(); plt.show()

---
## Exercises

### Exercise 1 -- Polygon area on sphere
Use the spherical excess formula: for a spherical polygon,
area = (sum of interior angles - (n-2)*pi) * R^2.

### Exercise 2 -- Geohash
Interleave lat and lon bits. For precision k, the geohash is a k-character base32 string.
Nearby locations share a common prefix.

### Exercise 3 -- R-tree
Extend the quadtree to store bounding boxes instead of points.
This is the foundation of PostGIS spatial indexing.

---
## Summary

| Concept | What it does | Standard / Library |
|---------|-------------|--------------------|
| WGS84 | Earth ellipsoid model | EPSG:4326, GPS |
| ECEF | Cartesian earth-centered coords | GPS hardware |
| Mercator | Conformal map projection | Google Maps, OSM |
| Haversine | Great-circle distance | geopy, turf.js |
| Quadtree | 2D spatial index | PostGIS, SpatiaLite |
| Slippy tiles | XYZ tile addressing | Leaflet, Mapbox, Cesium |

In [ ]:
print('Notebook complete.')
print('Key takeaways:')
print('  WGS84 ECEF   -> GPS latitude/lon to 3D Cartesian')
print('  Mercator     -> conformal, distorts area near poles')
print('  Haversine    -> great-circle distance formula')
print('  Quadtree     -> O(log N + k) spatial queries')
print('  Slippy tiles -> (z, x, y) tile addressing scheme')
print()
print('Production: GDAL, PostGIS, QGIS, Leaflet, Mapbox, Cesium, Google Earth Engine')